#

In [ ]:
!pip install faker

In [ ]:
from faker import Faker
import random
import pandas as pd
from datetime import datetime, timedelta, time

In [ ]:
fake = Faker('id_ID')

In [ ]:
def create_hospital_data(seed = 123):
  fake.seed_instance(seed)
  nama_rs = []
  for i in range (2):
    nama = fake.last_name()
    nama_rs.append('RS ' +fake.last_name())

  address_rs =[]
  for i in range (2):
    address = fake.address()
    address_rs.append(fake.address().replace("\n", ", "))

  hospital_dict = {'hospital_name':nama_rs, 'hospital_address': address_rs}
  data = pd.DataFrame(data=hospital_dict)

  return data

In [ ]:
hospitals = create_hospital_data()
hospitals

In [ ]:
def create_specializations(seed=123):
  fake.seed_instance(seed)
  specialization_name = ['General Practitioners', 'Pediatricians','Cardiologists','Dermatologists', 'Orthopedic']

  specialization_desc = fake.texts(nb_texts=5, max_nb_chars=50)

  spec_dict = {'specialization_name':specialization_name,
            'specialization _desc':specialization_desc}
  data = pd.DataFrame(data=spec_dict)

  return data

In [ ]:
specializations = create_specializations()
specializations

In [ ]:
def create_patients(seed=123, n=100):
  fake.seed_instance(seed)
  random.seed(seed)

  names = [fake.first_name() + " " + fake.last_name() for i in range (n)]

  gender_choices = ['Female','Male']
  genders = [random.choice(gender_choices) for i in range (n)]

  birthdates = [fake.date_of_birth(minimum_age = 15, maximum_age = 90) for i in range (n)]

  contacts = ['08'+ fake.numerify(text="##########") for i in range(n)]

  patient_dict = {'patient_name': names,
                    'patient_gender': genders,
                    'patient_birthdate': birthdates,
                    'patient_contact': contacts}
  data = pd.DataFrame(data = patient_dict)

  return data

In [ ]:
patients = create_patients()
patients

100 rows × 4 columns

In [ ]:
def create_doctors(seed=202, n=25, hospitals=None, specializations=None):
  fake.seed_instance(seed)
  random.seed(seed)
  names = [fake.first_name() + " " + fake.last_name() for i in range (n)]

  hospital_id = [random.randint(1, len(hospitals)) for i in range (n)]
  specialization_id = [random.randint(1, len(specializations)) for i in range (n)]

  doctors_dict = {'hospital_id':hospital_id,
                   'specialization_id': specialization_id,
                   'doctor_name': names}

  data = pd.DataFrame(doctors_dict)
  return data

In [ ]:
doctors = create_doctors(hospitals=hospitals, specializations=specializations)
doctors

In [ ]:
def create_doctor_schedule(doctors, seed=123):
    random.seed(seed)

    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    start_end_pairs = [
        (time(8, 0), time(12, 0)),
        (time(13, 0), time(17, 0)),
        (time(9, 0), time(13, 0)),
        (time(10, 0), time(14, 0))
    ]

    records = []

    for index, doctor in doctors.iterrows():
        doctor_id = index + 1

        # pilih 3 hari unik secara acak
        work_days = random.sample(days, 3)

        for day in work_days:
            start_time, end_time = random.choice(start_end_pairs)
            records.append({
                'doctor_id': doctor_id,
                'day_of_week': day,
                'start_time': start_time,
                'end_time': end_time
            })

    df = pd.DataFrame(records)
    return df

In [ ]:
doctor_schedule = create_doctor_schedule(doctors)
doctor_schedule

75 rows × 4 columns

In [ ]:
def create_appointment_slots(doctor_schedule, slot_duration_minutes=30):
    slots = []

    for index, sched in doctor_schedule.iterrows():
        start = sched['start_time']
        end = sched['end_time']
        current = datetime.combine(datetime.today(), start)

        while current.time() < end:
            slot_start = current.time()
            slot_end = (current + timedelta(minutes=slot_duration_minutes)).time()

            if slot_end > end:
                break

            slots.append({
                'schedule_id': index + 1,
                'slot_time_start': slot_start,
                'slot_time_end': slot_end
            })

            current += timedelta(minutes=slot_duration_minutes)

    return pd.DataFrame(slots)


In [ ]:
appointment_slots = create_appointment_slots(doctor_schedule)
appointment_slots

600 rows × 3 columns

In [ ]:
from datetime import datetime, timedelta
import random
import pandas as pd

def create_appointments(patients, appointment_slots, n=50, seed=123,
                        past_ratio=0.7, past_days=60, future_days=30):
    """
    Membuat dummy data untuk tabel appointments.

    Args:
        patients (DataFrame): tabel pasien.
        appointment_slots (DataFrame): tabel slot dokter.
        n (int): jumlah total appointment yang dihasilkan.
        seed (int): untuk reproducibility.
        past_ratio (float): proporsi appointment masa lalu.
        past_days (int): rentang hari ke belakang.
        future_days (int): rentang hari ke depan.

    Returns:
        DataFrame: appointments dengan kolom patient_id, slot_id, appointment_date, appointment_status.
    """
    random.seed(seed)

    appointments = []
    patient_ids = list(range(1, len(patients) + 1))
    slot_ids = list(range(1, len(appointment_slots) + 1))

    for _ in range(n):
        patient_id = random.choice(patient_ids)
        slot_id = random.choice(slot_ids)

        # tentukan apakah ini di masa lalu atau depan
        if random.random() < past_ratio:
            delta_days = -random.randint(1, past_days)  # mundur ke belakang
        else:
            delta_days = random.randint(1, future_days)  # ke depan

        appointment_date = datetime.today().date() + timedelta(days=delta_days)

        # tentukan status berdasarkan tanggal
        if appointment_date < datetime.today().date():
            status = random.choices(['Completed', 'Cancelled'], weights=[0.8, 0.2])[0]
        else:
            status = random.choices(['Scheduled', 'Cancelled'], weights=[0.9, 0.1])[0]

        appointments.append({
            'patient_id': patient_id,
            'slot_id': slot_id,
            'appointment_date': appointment_date,
            'appointment_status': status
        })

    # hapus duplikat slot_id + tanggal
    df = pd.DataFrame(appointments).drop_duplicates(subset=['slot_id', 'appointment_date'])
    return df.reset_index(drop=True)


In [ ]:
appointments = create_appointments(patients, appointment_slots)
appointments

In [ ]:
def create_medical_records(appointments):
    records = []

    # Filter hanya appointment yang sudah completed
    completed_appointments = appointments[appointments["appointment_status"] == "Completed"]

    for index, appt in completed_appointments.iterrows():
      records.append({
            "appointment_id": index + 1,
            "diagnosis": fake.text(max_nb_chars=50),   # kalimat acak
            "treatment": fake.text(max_nb_chars=60),
           })

    return pd.DataFrame(records)

In [ ]:
medical_records = create_medical_records(appointments)
medical_records

In [ ]:
patients.to_csv("patients.csv", index=False)
hospitals.to_csv("hospitals.csv", index=False)
specializations.to_csv("specializations.csv", index=False)
doctors.to_csv("doctors.csv", index=False)
doctor_schedule.to_csv("doctor_schedule.csv", index=False)
appointment_slots.to_csv("appointment_slots.csv", index=False)
appointments.to_csv("appointments.csv", index=False)
medical_records.to_csv("medical_records.csv", index=False)

In [ ]:
appointments.to_csv("appointments.csv", index=False)
medical_records.to_csv("medical_records.csv", index=False)